In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap
import pandas as pd
import os

In [3]:
dir_path = '../data processing/'
output_dir = "../dataset/"
map_paths = {
    'map90': dir_path + 'act-planck_dr4dr6_coadd_AA_daynight_f090_map.fits',
    'map150': dir_path + 'act-planck_dr4dr6_coadd_AA_daynight_f150_map.fits',
    'map220': dir_path + 'act-planck_dr6.02_coadd_AA_daynight_f220_map.fits'
}

df = pd.read_csv(dir_path + "clusters_selected.csv")

os.makedirs(output_dir, exist_ok=True)

In [4]:
len(df)

9977

In [ ]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from pixell import enmap
from tqdm import tqdm
import os

# === Предварительные расчёты ===
df["logM500c"] = np.log10(df["M500c"])
df["logM500cUncorr"] = np.log10(df["M500cUncorr"])

diameter_arcmin = df["theta500Arcmin"].max() * 2
width_deg = diameter_arcmin / 60.0
half = width_deg / 2.0

masses = df["logM500cUncorr"].values
top_15 = np.percentile(masses, 85)

# === Подготовим "чистые" данные для multiprocessing ===
rows = df[["RADeg", "decDeg", "redshift", "theta500Arcmin", "logM500c", "logM500cUncorr"]].to_dict(orient="records")

def process_row(row):
    try:
        ra = row["RADeg"]
        dec = row["decDeg"]
        z = row["redshift"]
        massa = row["logM500c"]
        massa_uncorr = row["logM500cUncorr"]

        box = np.deg2rad([
            [dec - half, ra - half],
            [dec + half, ra + half]
        ])
        pixbox = [[0, 0], [148, 148]]

        # === Пропускаем слабые массы ===
        if massa_uncorr < top_15:
            return None

        # === Читаем карты ===
        map90 = enmap.read_map(map_paths["map90"], box=box, pixbox=pixbox)[0]
        map150 = enmap.read_map(map_paths["map150"], box=box, pixbox=pixbox)[0]
        map220 = enmap.read_map(map_paths["map220"], box=box, pixbox=pixbox)[0]

        data_image = np.stack([map90, map150, map220]).astype(np.float32)
        results = []

        # === Аугментации ===
        # Для top_10 — повороты и отражения
        if massa_uncorr >= top_15:
            # 3 поворота
            for angle in [90, 180]:
                rotated = np.rot90(data_image, k=angle // 90, axes=(1, 2))
                results.append((rotated, z, massa, massa_uncorr))

            # 2 отражения
            flip_h = data_image[:, :, ::-1]
            flip_v = data_image[:, ::-1, :]
            r = np.random.randint(2)
            if r:
                results.append((flip_h, z, massa, massa_uncorr))
            else:
                results.append((flip_v, z, massa, massa_uncorr))
        return results

    except Exception as e:
        print(f"Ошибка при обработке (RA={row['RADeg']}, Dec={row['decDeg']}): {e}")
        return None


# === Параллельная обработка ===
all_images, all_redshifts, all_logM500c, all_logM500cUncorr = [], [], [], []

max_workers = min(4, os.cpu_count())  # ограничим, чтобы не перегружать систему
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    for result in tqdm(executor.map(process_row, rows), total=len(rows)):
        if result is not None:
            for img, z, m, mu in result:
                all_images.append(img)
                all_redshifts.append(z)
                all_logM500c.append(m)
                all_logM500cUncorr.append(mu)

# === Преобразуем в numpy массивы ===
images = np.array(all_images, dtype=np.float32)
redshifts = np.array(all_redshifts, dtype=np.float32)
logM500c = np.array(all_logM500c, dtype=np.float32)
logM500cUncorr = np.array(all_logM500cUncorr, dtype=np.float32)

print(f"✅ Обработано {len(images)} изображений (с аугментацией)")

100%|██████████| 9977/9977 [33:06<00:00,  5.02it/s]  


✅ Обработано 4497 изображений (с аугментацией)


In [8]:
for idx, image in enumerate(images):
    np.save(output_dir + f"images_augm/{idx+len(df)+1}.npy", image)

In [9]:
len(masses[masses >= top_15])

1499

In [10]:
len(df)

9977

In [ ]:
output_dir = "../dataset/"

In [11]:
redshifts.shape

(4497,)

In [13]:
redshifts_ = np.load("../dataset/redshifts.npy")
logM500c_ = np.load("../dataset/logM500c.npy")
logM500cUncorr_ = np.load("../dataset/logM500cUncorr.npy")

redsh = np.concatenate([redshifts_, redshifts])
M500c = np.concatenate([logM500c_, logM500c])
M500cUncorr = np.concatenate([logM500cUncorr_, logM500cUncorr])

In [14]:
np.save(output_dir + "redshifts_augm.npy", redsh)
np.save(output_dir + "logM500c_augm.npy", M500c)
np.save(output_dir + "logM500cUncorr_augm.npy", M500cUncorr)